# 02 · Embeddings
### Practical LLM Engineering — Module 01: Fundamentals

> **Objectives**
> - What embeddings are and the mathematical space they live in
> - How token embeddings, positional encodings, and sentence embeddings differ
> - How to compute and visualise similarity between embeddings
> - How embedding quality is measured and why it matters
> - Engineering insights: embedding dimensions, storage costs, retrieval tradeoffs

---


## 1. Overview

After tokenization, every token ID must be converted into a **dense vector** before the transformer can process it. That vector is called an **embedding**.

```
Token IDs:  [15496,  11,  995,  0]
                ↓  embedding lookup table  E ∈ ℝ^(V × d)
Embeddings: [ [0.12, -0.43, ..., 0.87],   # "Hello"
              [0.05,  0.21, ..., -0.33],   # ","
              [-0.71, 0.09, ..., 0.44],    # " world"
              [0.33, -0.12, ..., 0.91] ]   # "!"
              shape: (seq_len, d_model)
```

The key insight: **geometry encodes meaning**.

Words with similar meanings end up close together in this high-dimensional space. You can do arithmetic on meaning:

$$
\vec{\text{king}} - \vec{\text{man}} + \vec{\text{woman}} \approx \vec{\text{queen}}
$$

This notebook covers:
1. Token embeddings (lookup table inside every LLM)
2. Positional encodings (how position is injected)
3. Sentence / text embeddings (one vector per document)
4. Similarity measures
5. Dimensionality reduction for visualisation
6. Engineering considerations


## 2. Setup

In [ ]:
# Install dependencies (run once on Colab)
# !pip install transformers sentence-transformers scikit-learn matplotlib numpy torch -q

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from mpl_toolkits.mplot3d import Axes3D
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModel
from sentence_transformers import SentenceTransformer

print("Libraries ready")
print(f"   PyTorch  : {torch.__version__}")


## 3. Background: The Mathematics of Embeddings

### 3.1 What is an embedding?

An embedding is a function that maps a discrete symbol (token, word, sentence) into a continuous vector space $\mathbb{R}^d$:

$$
E: \mathcal{V} \rightarrow \mathbb{R}^d
$$

where $\mathcal{V}$ is the vocabulary and $d$ is the embedding dimension (e.g. 768, 1024, 4096).

The embedding **matrix** $\mathbf{E} \in \mathbb{R}^{|\mathcal{V}| \times d}$ is a learnable parameter.
For a token with index $i$, its embedding is simply row $i$ of $\mathbf{E}$:

$$
\mathbf{e}_i = \mathbf{E}[i, :] \in \mathbb{R}^d
$$

This is a **lookup operation** — no computation, just indexing.

---

### 3.2 Cosine Similarity

The standard way to compare two embeddings $\mathbf{u}, \mathbf{v} \in \mathbb{R}^d$:

$$
\text{cos\_sim}(\mathbf{u}, \mathbf{v}) = \frac{\mathbf{u} \cdot \mathbf{v}}{\|\mathbf{u}\| \cdot \|\mathbf{v}\|} \in [-1, 1]
$$

- $+1$: identical direction (semantically similar)
- $0$: orthogonal (unrelated)
- $-1$: opposite direction (antonyms, sometimes)

**Why cosine and not Euclidean?**
Embeddings from language models vary in magnitude. Cosine similarity normalises this out and focuses purely on **direction**, which captures semantic meaning more reliably.

---

### 3.3 Dot Product Similarity

Used in attention mechanisms and many retrieval systems:

$$
\text{sim}(\mathbf{q}, \mathbf{k}) = \mathbf{q} \cdot \mathbf{k} = \sum_{i=1}^{d} q_i k_i
$$

When vectors are L2-normalised, dot product = cosine similarity.


## 4. Token Embeddings — Inside an LLM

In [ ]:
# ── Build a minimal embedding layer ───────────────────────────────────
VOCAB_SIZE = 1000
D_MODEL    = 64   # small for illustration; real models use 768–8192

embedding_layer = nn.Embedding(VOCAB_SIZE, D_MODEL)
print(f"Embedding matrix shape : {embedding_layer.weight.shape}")
print(f"Parameters             : {embedding_layer.weight.numel():,}")
print(f"Memory (float32)       : {embedding_layer.weight.numel() * 4 / 1024:.1f} KB")
print()

# Lookup a few token embeddings
token_ids = torch.tensor([42, 100, 500])
embeddings = embedding_layer(token_ids)
print(f"Token IDs : {token_ids.tolist()}")
print(f"Embeddings shape: {embeddings.shape}  (seq_len=3, d_model={D_MODEL})")
print(f"First token embedding (first 8 dims):\n  {embeddings[0, :8].detach().numpy().round(3)}")


In [ ]:
# ── Real model: extract token embeddings from GPT-2 ──────────────────
from transformers import GPT2Model, GPT2Tokenizer

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model     = GPT2Model.from_pretrained("gpt2")
model.eval()

E = model.wte.weight  # Word Token Embeddings: shape (50257, 768)
print(f"GPT-2 embedding matrix : {E.shape}")
print(f"  Vocab size           : {E.shape[0]:,}")
print(f"  Embedding dim (d)    : {E.shape[1]}")
print(f"  Memory (float32)     : {E.numel() * 4 / 1024 / 1024:.1f} MB")

# Compare two token embeddings
token_a = tokenizer.encode("king")[0]
token_b = tokenizer.encode("queen")[0]
token_c = tokenizer.encode("dog")[0]

ea = E[token_a].detach()
eb = E[token_b].detach()
ec = E[token_c].detach()

def cosine_sim(a, b):
    return (a @ b / (a.norm() * b.norm())).item()

print(f"\nToken similarity (raw token embeddings):")
print(f"  cos_sim('king',  'queen') = {cosine_sim(ea, eb):.4f}")
print(f"  cos_sim('king',  'dog')   = {cosine_sim(ea, ec):.4f}")
print(f"  cos_sim('queen', 'dog')   = {cosine_sim(eb, ec):.4f}")


## 5. Positional Encodings

Transformers are **permutation-invariant** — without positional information, the sequence
`["cat", "sat", "the"]` looks identical to `["the", "cat", "sat"]`.

Positional encodings inject position information by **adding** a positional vector to each token embedding:

$$
\mathbf{x}_i = \mathbf{e}_i + \mathbf{p}_i
$$

### 5.1 Sinusoidal Positional Encoding (original Transformer)

$$
\text{PE}(pos, 2i)   = \sin\!\left(\frac{pos}{10000^{2i/d}}\right)
$$
$$
\text{PE}(pos, 2i+1) = \cos\!\left(\frac{pos}{10000^{2i/d}}\right)
$$

where $pos$ is the position index and $i$ is the dimension index.

**Why sinusoids?**
- Fixed (not learned) — works at any sequence length
- Smooth variation across positions
- The relative distance between positions can be expressed as a linear transformation


In [ ]:
def sinusoidal_positional_encoding(seq_len: int, d_model: int) -> np.ndarray:
    """Compute sinusoidal positional encodings as in Vaswani et al. (2017)."""
    PE = np.zeros((seq_len, d_model))
    positions = np.arange(seq_len)[:, np.newaxis]          # (seq_len, 1)
    div_term  = np.power(10000, np.arange(0, d_model, 2) / d_model)  # (d_model/2,)

    PE[:, 0::2] = np.sin(positions / div_term)   # even dims → sin
    PE[:, 1::2] = np.cos(positions / div_term)   # odd  dims → cos
    return PE


# ── Visualise ──────────────────────────────────────────────────────────
seq_len = 64
d_model = 128
PE = sinusoidal_positional_encoding(seq_len, d_model)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Heatmap of PE matrix
im = axes[0].imshow(PE, aspect="auto", cmap="RdBu", vmin=-1, vmax=1)
axes[0].set_xlabel("Embedding dimension")
axes[0].set_ylabel("Position")
axes[0].set_title("Sinusoidal PE — full matrix")
plt.colorbar(im, ax=axes[0])

# Individual dimensions over position
for dim in [0, 1, 4, 8, 16, 32]:
    axes[1].plot(PE[:, dim], label=f"dim {dim}", alpha=0.8)
axes[1].set_xlabel("Position")
axes[1].set_ylabel("Value")
axes[1].set_title("PE values per dimension across positions")
axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Key property: lower dimensions oscillate faster (fine-grained position),")
print(f"higher dimensions oscillate slower (coarse-grained position).")


### 5.2 Rotary Positional Encoding (RoPE)

Modern LLMs (LLaMA, Mistral, GPT-NeoX) use **RoPE** instead of sinusoidal PE.

RoPE encodes position by **rotating** query and key vectors in 2D subspaces before attention:

$$
\mathbf{q}_m^\top \mathbf{k}_n = \text{Re}\left[\sum_{j=0}^{d/2-1} q_{m,j} k_{n,j}^* e^{i(m-n)\theta_j}\right]
$$

The dot product depends only on the **relative position** $(m - n)$, not absolute positions.

**Advantages over sinusoidal PE:**
- Better length extrapolation
- Relative position is naturally encoded
- Can be extended to longer contexts (e.g. YaRN, LongRoPE)


In [ ]:
def rope_rotation(x: torch.Tensor, position: int, theta_base: float = 10000.0) -> torch.Tensor:
    """Apply RoPE rotation to a vector x at a given position."""
    d = x.shape[-1]
    assert d % 2 == 0
    half_d = d // 2

    # Compute rotation angles for each dimension pair
    i = torch.arange(half_d, dtype=torch.float32)
    theta = position / (theta_base ** (2 * i / d))  # shape: (half_d,)

    cos_theta = torch.cos(theta)
    sin_theta = torch.sin(theta)

    x1, x2 = x[..., :half_d], x[..., half_d:]
    x_rotated = torch.cat([
        x1 * cos_theta - x2 * sin_theta,
        x1 * sin_theta + x2 * cos_theta
    ], dim=-1)
    return x_rotated


# Demonstrate: same query at different positions
d = 64
query = torch.randn(d)

positions = [0, 1, 10, 50, 100]
rotated = [rope_rotation(query, pos) for pos in positions]

# Dot products between position 0 and all others (should decay with distance)
base = rotated[0]
print("RoPE: dot product of query@pos=0 with query@pos=k")
print("(captures relative position — values decay with distance)")
print()
for pos, rot in zip(positions, rotated):
    dp = (base @ rot).item()
    print(f"  pos={pos:4d}  dot_product={dp:.4f}")


## 6. Sentence Embeddings

Token embeddings represent individual tokens. For retrieval and semantic search, we need **one vector per sentence or document**.

### How are sentence embeddings produced?

Three common strategies:

| Strategy | Description | Used in |
|---|---|---|
| **[CLS] token** | Use the first token's final hidden state | BERT, RoBERTa |
| **Mean pooling** | Average all token hidden states | Sentence-BERT, E5 |
| **Last token** | Use the last token (EOS) hidden state | GPT-style decoder models |

**Mean pooling** is typically best:

$$
\mathbf{s} = \frac{1}{n} \sum_{i=1}^{n} \mathbf{h}_i
$$

where $\mathbf{h}_i$ is the final-layer hidden state for token $i$.


In [ ]:
# ── Mean-pooling from a raw transformer ───────────────────────────────
from transformers import AutoTokenizer, AutoModel
import torch

model_name = "sentence-transformers/all-MiniLM-L6-v2"
st_tokenizer = AutoTokenizer.from_pretrained(model_name)
st_model     = AutoModel.from_pretrained(model_name)
st_model.eval()

def mean_pool(model_output, attention_mask):
    """Mean pooling over token hidden states, respecting padding."""
    token_embeddings = model_output.last_hidden_state  # (batch, seq, d)
    mask = attention_mask.unsqueeze(-1).float()        # (batch, seq, 1)
    summed = (token_embeddings * mask).sum(dim=1)      # (batch, d)
    counts = mask.sum(dim=1).clamp(min=1e-9)           # (batch, 1)
    return summed / counts                             # (batch, d)

def encode(texts: list[str]) -> np.ndarray:
    inputs = st_tokenizer(texts, padding=True, truncation=True,
                          max_length=256, return_tensors="pt")
    with torch.no_grad():
        outputs = st_model(**inputs)
    embeddings = mean_pool(outputs, inputs["attention_mask"])
    # L2 normalise
    embeddings = torch.nn.functional.normalize(embeddings, p=2, dim=1)
    return embeddings.numpy()

# Test
sentences = [
    "The cat sat on the mat.",
    "A dog lay on the rug.",
    "Machine learning models require training data.",
    "Deep neural networks learn representations.",
    "Paris is the capital of France.",
    "Berlin is Germany's capital city.",
]

embeddings = encode(sentences)
print(f"Embeddings shape: {embeddings.shape}  ({len(sentences)} sentences × {embeddings.shape[1]} dims)")


In [ ]:
# ── Full similarity matrix ─────────────────────────────────────────────
similarity_matrix = embeddings @ embeddings.T  # cosine sim since L2-normalised

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(similarity_matrix, cmap="Blues", vmin=0, vmax=1)
plt.colorbar(im, ax=ax, label="Cosine similarity")

short_labels = [s[:30] + "…" if len(s) > 30 else s for s in sentences]
ax.set_xticks(range(len(sentences)))
ax.set_yticks(range(len(sentences)))
ax.set_xticklabels(short_labels, rotation=45, ha="right", fontsize=8)
ax.set_yticklabels(short_labels, fontsize=8)
ax.set_title("Sentence Embedding Similarity Matrix")

# Annotate cells
for i in range(len(sentences)):
    for j in range(len(sentences)):
        ax.text(j, i, f"{similarity_matrix[i,j]:.2f}",
                ha="center", va="center", fontsize=8,
                color="black" if similarity_matrix[i,j] < 0.7 else "white")

plt.tight_layout()
plt.show()


## 7. Visualising Embeddings

Embeddings live in hundreds of dimensions — impossible to visualise directly.
Two standard techniques project them to 2D/3D:

| Method | Type | Best for |
|---|---|---|
| **PCA** | Linear | Global structure, fast, deterministic |
| **t-SNE** | Non-linear | Local clustering, exploratory analysis |
| **UMAP** | Non-linear | Both global & local, fast |

We'll use PCA and t-SNE on a richer sentence set.


In [ ]:
# ── Build a richer dataset ─────────────────────────────────────────────
categories = {
    "Animals":    ["The lion is the king of the jungle.",
                   "Dogs are loyal companions.",
                   "Cats are independent animals.",
                   "Eagles soar high in the sky."],
    "Technology": ["Machine learning requires large datasets.",
                   "Neural networks have many layers.",
                   "GPUs accelerate deep learning training.",
                   "Transformers revolutionised NLP."],
    "Geography":  ["Paris is the capital of France.",
                   "Tokyo is the largest city in Japan.",
                   "The Amazon river flows through Brazil.",
                   "Mount Everest is the tallest peak."],
    "Food":       ["Pizza is popular in Italy.",
                   "Sushi is a Japanese delicacy.",
                   "Tacos are a Mexican dish.",
                   "Pasta comes in many shapes."],
}

all_sentences = []
all_labels    = []
for cat, sents in categories.items():
    all_sentences.extend(sents)
    all_labels.extend([cat] * len(sents))

all_embeddings = encode(all_sentences)
print(f"Dataset: {len(all_sentences)} sentences across {len(categories)} categories")
print(f"Embedding matrix: {all_embeddings.shape}")


In [ ]:
# ── PCA projection ────────────────────────────────────────────────────
pca = PCA(n_components=2)
pca_2d = pca.fit_transform(all_embeddings)

print(f"PCA explained variance: {pca.explained_variance_ratio_[:2].sum()*100:.1f}%")

# ── t-SNE projection ──────────────────────────────────────────────────
tsne = TSNE(n_components=2, perplexity=5, random_state=42, n_iter=1000)
tsne_2d = tsne.fit_transform(all_embeddings)

# ── Plot both ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
color_map = {"Animals": "#e74c3c", "Technology": "#3498db",
             "Geography": "#2ecc71", "Food": "#f39c12"}

for ax, coords, title in [(axes[0], pca_2d, "PCA"), (axes[1], tsne_2d, "t-SNE")]:
    for i, (x, y) in enumerate(coords):
        cat = all_labels[i]
        ax.scatter(x, y, color=color_map[cat], s=100, zorder=3)
        ax.annotate(all_sentences[i][:25] + "…",
                    (x, y), textcoords="offset points",
                    xytext=(5, 5), fontsize=7, alpha=0.8)
    # Legend
    for cat, col in color_map.items():
        ax.scatter([], [], color=col, label=cat, s=80)
    ax.legend(fontsize=9)
    ax.set_title(f"{title} projection of sentence embeddings")
    ax.grid(alpha=0.3)
    ax.set_xlabel("Component 1")
    ax.set_ylabel("Component 2")

plt.suptitle("Semantically similar sentences cluster together", fontsize=12, y=1.02)
plt.tight_layout()
plt.show()


## 8. Engineering Insights

### 8.1 Embedding Dimension vs Storage Cost

The embedding dimension $d$ is a critical design decision that trades off **expressiveness** against **cost**.

| Model | $d$ | Embedding matrix size (float32) |
|---|---|---|
| MiniLM-L6 | 384 | small |
| BERT-base | 768 | ~38 MB |
| GPT-2 | 768 | ~154 MB |
| LLaMA-7B | 4096 | ~800 MB |
| GPT-4 (est.) | ~12288 | >1 GB |

For retrieval systems, storing millions of embeddings:

$$
\text{storage} = N \times d \times \text{bytes\_per\_float}
$$

e.g. 10M documents × 768 dims × 4 bytes = **30.7 GB** (float32)


In [ ]:
# ── Storage cost calculator ───────────────────────────────────────────
def embedding_storage_cost(n_docs: int, d: int, dtype_bytes: int = 4) -> dict:
    total_bytes  = n_docs * d * dtype_bytes
    total_MB     = total_bytes / 1024**2
    total_GB     = total_bytes / 1024**3
    return {"n_docs": n_docs, "dim": d, "bytes": total_bytes,
            "MB": total_MB, "GB": total_GB}

configs = [
    (10_000,     384, "10K docs, MiniLM"),
    (100_000,    768, "100K docs, BERT"),
    (1_000_000,  768, "1M docs, BERT"),
    (1_000_000, 1536, "1M docs, text-embedding-3-large"),
    (10_000_000, 768, "10M docs, BERT"),
]

print(f"{'Config':<40} {'Dims':>5} {'Size':>12}")
print("-" * 60)
for n, d, label in configs:
    c = embedding_storage_cost(n, d)
    size_str = f"{c['GB']:.2f} GB" if c['GB'] >= 1 else f"{c['MB']:.1f} MB"
    print(f"{label:<40} {d:>5} {size_str:>12}")


### 8.2 Quantisation of Embeddings

Float32 embeddings are expensive. Common optimisations:

| Precision | Bytes/dim | Storage vs FP32 | Quality loss |
|---|---|---|---|
| float32 | 4 | 1× (baseline) | none |
| float16 | 2 | 0.5× | minimal |
| int8 | 1 | 0.25× | small |
| binary | 0.125 | 0.03× | moderate |

Binary embeddings (1 bit per dim) reduce storage by **32×** with only ~10–15% recall loss in practice — useful for very large retrieval systems.


In [ ]:
# ── Quantisation experiment ───────────────────────────────────────────
query   = all_embeddings[0]    # "The lion is the king of the jungle."
targets = all_embeddings[1:]

def cosine_batch(q, targets):
    return (targets @ q) / (np.linalg.norm(targets, axis=1) * np.linalg.norm(q) + 1e-9)

# float32 baseline
sims_f32 = cosine_batch(query, targets)

# float16
q_f16, t_f16 = query.astype(np.float16), targets.astype(np.float16)
sims_f16 = cosine_batch(q_f16.astype(np.float32), t_f16.astype(np.float32))

# int8 (simple linear quantisation)
def quantize_int8(x):
    scale = np.max(np.abs(x)) / 127
    return np.clip(np.round(x / scale), -128, 127).astype(np.int8), scale

q_i8, qs = quantize_int8(query)
t_i8 = np.array([quantize_int8(t)[0] for t in targets])
sims_i8 = cosine_batch(q_i8.astype(np.float32), t_i8.astype(np.float32))

# Binary
q_bin = (query > 0).astype(np.float32) * 2 - 1
t_bin = (targets > 0).astype(np.float32) * 2 - 1
sims_bin = cosine_batch(q_bin, t_bin)

print(f"{'Sentence':<45} {'F32':>6} {'F16':>6} {'INT8':>6} {'BIN':>6}")
print("-" * 70)
for i, sent in enumerate(all_sentences[1:]):
    print(f"{sent[:43]:<45} {sims_f32[i]:>6.3f} {sims_f16[i]:>6.3f} "
          f"{sims_i8[i]:>6.3f} {sims_bin[i]:>6.3f}")


### 8.3 Choosing an Embedding Model

Key considerations when selecting an embedding model for a system:

| Factor | Consideration |
|---|---|
| **Dimension** | Higher = more expressive, higher cost |
| **Max input tokens** | Critical for long documents (e.g. 512 vs 8192) |
| **Task** | Retrieval vs classification vs clustering differ |
| **Language** | Multilingual vs English-only |
| **Speed** | MiniLM (fast) vs large models (slow) |
| **MTEB score** | Standard benchmark — check [huggingface.co/spaces/mteb/leaderboard](https://huggingface.co/spaces/mteb/leaderboard) |

**Rule of thumb:**
- For RAG / retrieval: `text-embedding-3-small` (API) or `BAAI/bge-small-en-v1.5` (local)
- For multilingual: `intfloat/multilingual-e5-base`
- For constrained environments: `all-MiniLM-L6-v2` (22M params, 384 dims)


## 9. The Geometry of Meaning

In [ ]:
# ── Analogy test: king - man + woman ≈ queen ─────────────────────────
# (Works better with large models — using sentence-transformers here for demo)
analogy_words = ["king", "man", "woman", "queen",
                 "paris", "france", "berlin", "germany",
                 "fast", "slow", "hot", "cold"]

word_embeddings = encode(analogy_words)
word_map = {w: word_embeddings[i] for i, w in enumerate(analogy_words)}

def analogy(a, b, c, word_map, top_k=3):
    """a - b + c = ?"""
    target = word_map[a] - word_map[b] + word_map[c]
    target /= np.linalg.norm(target)
    sims = {w: float(target @ v) for w, v in word_map.items()
            if w not in {a, b, c}}
    return sorted(sims.items(), key=lambda x: -x[1])[:top_k]

print("Analogy: A - B + C → ?")
print("=" * 40)
for (a, b, c, expected) in [("king", "man", "woman", "queen"),
                              ("paris", "france", "berlin", "germany")]:
    results = analogy(a, b, c, word_map)
    print(f"  {a} - {b} + {c} →  {results}")
    print(f"  Expected: {expected}")
    print()


## 10. Exercises

1. **Pooling strategies:** Implement CLS-token pooling and last-token pooling. Compare their similarity matrices against mean pooling. Which gives better semantic clustering?

2. **Quantisation recall:** Build a small retrieval system (50 sentences). Measure top-5 recall using float32 vs int8 vs binary embeddings. At what compression level does quality degrade significantly?

3. **Positional encoding length:** Extend the sinusoidal PE to `seq_len=2048`. Does the pattern remain well-behaved? Now compute the similarity between PE vectors at positions 0 and 1, 0 and 100, 0 and 1000. What do you notice?

4. **Multilingual geometry:** Encode "cat", "gato" (Spanish), "chat" (French), "猫" (Chinese) using a multilingual model. Are they close together in embedding space? What does this imply for cross-lingual retrieval?

5. **Embedding drift:** Encode the sentence "I went to the bank" (financial) and "I sat by the river bank" (geographical). Are the embeddings similar or different? What does this tell you about contextual vs static embeddings?

---

## 11. Key Takeaways

| Concept | Summary |
|---|---|
| **Embedding** | Dense vector representing a token, word, or sentence in $\mathbb{R}^d$ |
| **Token embedding** | Row lookup in matrix $\mathbf{E} \in \mathbb{R}^{V \times d}$ |
| **Positional encoding** | Injected to give the model position awareness |
| **RoPE** | Modern PE — encodes relative position via rotation |
| **Sentence embedding** | Single vector per text, usually via mean pooling |
| **Cosine similarity** | Standard measure; normalises magnitude |
| **Quantisation** | int8/binary reduces storage 4–32× with small quality loss |
| **Dimension choice** | Higher $d$ → richer representation, higher storage & compute cost |
